# Ingesta incremental de mediciones a Bronze

Copiar los CSV de un lote a la carpeta `mediciones_input_dir` indicada en `config/bronze.json` y ejecutar el notebook. Auto Loader registra los archivos procesados en el checkpoint; repetir la ejecución no vuelve a insertar esos archivos. No limpiar el checkpoint entre lotes.


In [ ]:
import csv
import json
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

config_path = next((parent / 'config' / 'bronze.json' for parent in (Path.cwd(), *Path.cwd().parents) if (parent / 'config' / 'bronze.json').is_file()), None)
if config_path is None:
    raise FileNotFoundError('No se encuentra config/bronze.json desde el directorio del notebook')
config = json.loads(config_path.read_text(encoding='utf-8'))
INPUT_DIR = config['mediciones_input_dir']
TARGET_TABLE = config['tables']['mediciones']
CHECKPOINT = config['checkpoints']['mediciones']
expected_columns = list(config['input_schemas']['mediciones'])
assert all(dtype.lower() == 'string' for dtype in config['input_schemas']['mediciones'].values())
SOURCE_SCHEMA = StructType([StructField(name, StringType(), True) for name in expected_columns])
print(INPUT_DIR, TARGET_TABLE, CHECKPOINT, sep='\n')


## Validación previa

Se comprueba la cabecera de todos los CSV presentes antes de iniciar la escritura. No sobrescribir archivos ya ingeridos: una corrección debe llegar con otro nombre de archivo.


In [ ]:
files = [entry.path for entry in dbutils.fs.ls(INPUT_DIR) if entry.path.lower().endswith('.csv')]
assert files, f'No hay CSV en {INPUT_DIR}'
for file_path in files:
    header_line = spark.read.text(file_path).first()[0].lstrip('\ufeff')
    actual_columns = next(csv.reader([header_line]))
    assert actual_columns == expected_columns, f'Cabecera inesperada en {file_path}: {actual_columns}; esperada: {expected_columns}'
print(f'Cabeceras verificadas: {len(files)}')


## Carga

Las ocho columnas de origen siguen siendo cadenas. `_source_file` conserva la ruta de procedencia para las validaciones posteriores en Silver.


In [ ]:
mediciones = (spark.readStream.format('cloudFiles')
    .option('cloudFiles.format', 'csv')
    .option('header', 'true')
    .option('mode', 'FAILFAST')
    .option('ignoreLeadingWhiteSpace', 'false')
    .option('ignoreTrailingWhiteSpace', 'false')
    .schema(SOURCE_SCHEMA)
    .load(INPUT_DIR)
    .withColumn('_source_file', F.col('_metadata.file_path')))

query = (mediciones.writeStream.format('delta')
    .option('checkpointLocation', CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE))
query.awaitTermination()


In [ ]:
resultado = spark.table(TARGET_TABLE)
assert all(field.dataType == StringType() for field in resultado.schema if field.name != '_source_file')
assert resultado.filter(F.col('_source_file').isNull()).count() == 0
print('Filas acumuladas en Bronze:', resultado.count())
display(resultado.groupBy('_source_file').count().orderBy('_source_file'))
